In [2]:
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


In [2]:

import pandas as pd


df = pd.read_csv('Ev_battery.csv')

In [3]:
df.head()

In [4]:
df.columns

In [5]:
df['battery_failure'].value_counts()

In [6]:
df.info()

In [7]:
df.T.duplicated().sum()

In [8]:
df.duplicated().sum()

In [9]:
df.columns[df.isna().sum()>0]

In [10]:
df.drop(['vehicle_id','battery_serial'],axis=1,inplace=True)

In [11]:
df.corr(numeric_only=True)['battery_failure'].sort_values(ascending=False)

In [12]:
X=df.drop('battery_failure',axis=1)
y=df['battery_failure']

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [14]:
num=X_train.select_dtypes(include=['int64','float64'])
cat=X_train.select_dtypes(include='object')

In [22]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [23]:
cat_pipeline=Pipeline([ 
    ('cat',SimpleImputer(strategy='most_frequent')),
    ('ohe',OneHotEncoder(handle_unknown='ignore'))])

In [24]:
trf1=ColumnTransformer([
    ('num',SimpleImputer(strategy='median'),num.columns),
    ('cat',cat_pipeline,cat.columns),
])

In [ ]:
cat

In [25]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
#trf2=ColumnTransformer([
#    ('ohe',OneHotEncoder(handle_unknown='ignore'),cat.columns)
#])

In [26]:
from sklearn.feature_selection import mutual_info_classif,SelectKBest
mi_selector = SelectKBest(score_func=mutual_info_classif,k=30)

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
rf_selector=SelectFromModel(
    RandomForestClassifier(random_state=42),
    threshold='median'
)

rf_model = RandomForestClassifier(random_state=42)

In [28]:
from sklearn.pipeline import Pipeline

In [29]:
pipe = Pipeline([
    ('trf1',trf1),
    ('mutual_info',mi_selector),
    ('rf_selector',rf_selector),
    ('model',rf_model)
])

In [40]:
pipe.fit(X_train,y_train)

In [48]:
pipe.set_params(
    model__max_depth=10,
    model__min_samples_split=8,
    model__min_samples_leaf=3
)

In [49]:
pipe.fit(X_train,y_train)

In [50]:
pipe.score(X_train,y_train)

In [51]:
pipe.score(X_test,y_test)

In [ ]:
features = pipe.named_steps["trf1"].get_feature_names_out()
mi_features = features[
    pipe.named_steps["mutual_info"].get_support()
]
final_features = mi_features[
    pipe.named_steps["rf_selector"].get_support()
]
print("Final Features ", final_features)


In [57]:
import joblib
joblib.dump(pipe,'ev_battery_model.pkl')